In [1]:
import pandas as pd
import great_expectations as gx
import time

def run_stage1_validation(data_path):
    print(f"\n==========================================")
    print(f"STAGE 1 VALIDATION (Great Expectations) ON: {data_path}")
    print(f"==========================================")

    df_raw = pd.read_csv(data_path)
    df_filtered = df_raw[df_raw['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

    t_start_valid = time.time()

    # 1. Ephemeral context
    context = gx.get_context(mode="ephemeral")

    # 2. Pandas datasource + dataframe asset
    #    NOTE: no `dataframe=` here in GX >= 0.16.15
    datasource = context.data_sources.add_pandas(name="paysim_datasource")
    asset = datasource.add_dataframe_asset(name="paysim_frame")

    # 3. Batch definition
    batch_definition = asset.add_batch_definition_whole_dataframe(
        name="paysim_batch"
    )

    # 4. Pass the DataFrame HERE (this is the new location)
    batch = batch_definition.get_batch(
        batch_parameters={"dataframe": df_filtered}
    )

    # 5. Validator bound to the batch
    suite = context.suites.add(
        gx.ExpectationSuite(name="paysim_ingestion_suite")
    )
    validator = context.get_validator(
        batch=batch,
        expectation_suite=suite
    )

    # 6. Declare expectations
    target_cols = [
        'amount', 'oldbalanceOrg', 'newbalanceOrig',
        'oldbalanceDest', 'newbalanceDest'
    ]

    for col in target_cols:
        validator.expect_column_values_to_not_be_null(column=col)

    validator.expect_column_values_to_be_between(
        column='amount', min_value=0
    )

    validator.expect_column_to_exist(column='oldbalanceOrg')
    validator.expect_column_to_exist(column='newbalanceOrig')

    # 7. Validate
    validation_results = validator.validate()
    t_end_valid = time.time()

    # 8. Metrics
    t_valid_duration = t_end_valid - t_start_valid
    success = validation_results["success"]
    stats = validation_results["statistics"]

    print(f"Validation Status: {'SUCCESS (Pristine Data)' if success else 'FAILED (Anomalies Detected)'}")
    print(f"Validation Time (T_valid): {t_valid_duration:.4f} seconds")
    print(f"Evaluated Rules: {stats['evaluated_expectations']}")
    print(f"Passed Rules: {stats['successful_expectations']}")
    print(f"Failed Rules: {stats['unsuccessful_expectations']}")

    return validator, validation_results, t_valid_duration


if __name__ == "__main__":
    validator, results, t_valid = run_stage1_validation('Dataset.csv')


STAGE 1 VALIDATION (Great Expectations) ON: Dataset.csv


Calculating Metrics: 100%|██████████| 21/21 [00:00<00:00, 27.14it/s] 


Validation Status: SUCCESS (Pristine Data)
Validation Time (T_valid): 11.0559 seconds
Evaluated Rules: 7
Passed Rules: 7
Failed Rules: 0


In [2]:
if __name__ == "__main__":
    noisy_files = [
        'paysim_noise_5.csv',
        'paysim_noise_10.csv',
        'paysim_noise_20.csv',
    ]

    summary = []

    for path in noisy_files:
        validator, results, t_valid = run_stage1_validation(path)

        # Extract failed expectations from the GX result object
        errors = []
        for result in results.results:
            if not result.success:
                # result.expectation_config contains the expectation type and parameters
                expectation_type = result.expectation_config.type
                column = result.expectation_config.kwargs.get('column', 'N/A')
                errors.append(f"{expectation_type} failed on column '{column}'")

        summary.append({
            'file': path,
            'success': results.success,
            'errors': errors,
            't_valid_sec': round(t_valid, 4),
        })

    # Print a compact comparison table
    print("\n==========================================")
    print("NOISE TEST SUMMARY")
    print("==========================================")
    for row in summary:
        print(f"{row['file']:<25} | success={row['success']!s:<5} | "
              f"errors={row['errors']} | T_valid={row['t_valid_sec']}s")


STAGE 1 VALIDATION (Great Expectations) ON: paysim_noise_5.csv


Calculating Metrics: 100%|██████████| 21/21 [00:01<00:00, 20.88it/s] 


Validation Status: FAILED (Anomalies Detected)
Validation Time (T_valid): 9.4115 seconds
Evaluated Rules: 7
Passed Rules: 2
Failed Rules: 5

STAGE 1 VALIDATION (Great Expectations) ON: paysim_noise_10.csv


Calculating Metrics: 100%|██████████| 21/21 [00:01<00:00, 12.62it/s]


Validation Status: FAILED (Anomalies Detected)
Validation Time (T_valid): 11.8959 seconds
Evaluated Rules: 7
Passed Rules: 2
Failed Rules: 5

STAGE 1 VALIDATION (Great Expectations) ON: paysim_noise_20.csv


Calculating Metrics: 100%|██████████| 21/21 [00:01<00:00, 15.51it/s]


Validation Status: FAILED (Anomalies Detected)
Validation Time (T_valid): 11.7134 seconds
Evaluated Rules: 7
Passed Rules: 2
Failed Rules: 5

NOISE TEST SUMMARY
paysim_noise_5.csv        | success=False | errors=["expect_column_values_to_not_be_null failed on column 'amount'", "expect_column_values_to_not_be_null failed on column 'oldbalanceOrg'", "expect_column_values_to_not_be_null failed on column 'newbalanceOrig'", "expect_column_values_to_not_be_null failed on column 'oldbalanceDest'", "expect_column_values_to_not_be_null failed on column 'newbalanceDest'"] | T_valid=9.4115s
paysim_noise_10.csv       | success=False | errors=["expect_column_values_to_not_be_null failed on column 'amount'", "expect_column_values_to_not_be_null failed on column 'oldbalanceOrg'", "expect_column_values_to_not_be_null failed on column 'newbalanceOrig'", "expect_column_values_to_not_be_null failed on column 'oldbalanceDest'", "expect_column_values_to_not_be_null failed on column 'newbalanceDest'"] | T_v